In [1]:
# GLOBAL #

import random
import pandas as pd
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import numpy as np
from itertools import product
from typing import Literal
import os
import string

random.seed(123)
np.random.seed(123)
random_state = 123

Way  = Literal['lagged', 'semicol']

# 1. Markov Sampling Validation

- How well can Variable Order Markov (VOM) sampling methods capture the rules used to generate data?
- Under what data conditions do these methods work?
- Do my actual datasets meet these conditions?

In [2]:
# HELPER FUNCTIONS # 

# collapse sequence by self-transitions (e.g. [aaaabbbbcccdddccc] = [abcdc]) for collapsed context model
def collapseLength(seq):

    collapsed = []

    for s in seq:
        if not collapsed or collapsed[-1] != s:
            collapsed.append(s)

    return collapsed

# check if tuple has consecutive repeated states (we want to ignore these combinations in collapsed context model transition matrix)
def hasConsecutiveRepeats(states):

    for i in range(len(states) - 1):
        if states[i] == states[i + 1]:
            return True
        
    return False

# find indices of first state in each state run within sequence (e.g., [aaaabbbbcccdddccccc] = [0, 4, 8, 11, 14])
def uniqueIdxs(seq):

    lasts = []
    runChar = seq[0]

    for i, s in enumerate(seq[1:], start=1):
        if s != runChar:
            lasts.append(i - 1)
            runChar = s

    lasts.append(len(seq) - 1)
    return lasts

# expand  numeric vector into repetitions corresponding to each number value (e.g., [0, 4, 8, 11, 14] = [0, 0, 0, 0, 4, 4, 4, 4, 8, 8, 8, 11, 11, 11, 14, 14, 14]) 
# for matching strings to indices in collapsed context model 
def expandVec(vec):

    if not vec:
        return []
    
    result = []

    vecIDX = 0

    for pos in range(vec[-1] + 1):
        while vecIDX < len(vec) - 1 and pos > vec[vecIDX]:
            vecIDX += 1
        result.append(vec[vecIDX])

    return result

#calculate weighted log likelihood of test transitions (unnormalized) under train model (normalized)
def averageLL(trainProbs, testProbs):

    ll = 0.0 # start at zero

    weight = 0.0

    for s, nexts in testProbs.items(): #for each item in the test transition 
        trainContext = trainProbs.get(s) #find matching train model probability

        if trainContext is None:
            continue  
                             # if context unseen in train, skip
        for s2, p_test in nexts.items():
            p_train = trainContext.get(s2)

            if p_train is None or p_train <= 0:
                continue  # if transition unseen in train, skip
                             
            ll += p_test * np.log(p_train)
            weight += p_test

    if weight == 0:
        return np.nan
    
    return ll / weight

#### main analysis functions ####

""" pipeline:

normalize the training set because we are essentially building a smoothed model that we can use to evaluate test set fit
keep test set as count data because we are comparing how well it fits to our smoothed model

 """

##### Lagged Context Markov Sampling #####

def laggedTransNorm(states, lines, k, smoother, lag):

    transitions = defaultdict(Counter)
    
    #for k=0 version, transition matrix turns into simple marginal state distributions
    if k == 0:
        for state in states:
            transitions[()][state] = smoother #add laplace smoother to each curr state

        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

        total = sum(transitions[()].values())

        return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

    for prev in product(states, repeat=k): #add laplace smoother to every combination of prev & curr states

        for curr in states:
            transitions[prev][curr] = smoother

    for seq in lines:
        seqLength = len(seq)

        for t in range(k * lag, seqLength):
            prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
            curr = seq[t] # curr state is at time t
            transitions[prev][curr] += 1 #add 1 count to this prev & curr combination

    transNorm = {}

    for cond, counter in transitions.items():
        total = sum(counter.values())
        transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize counts by total number of transitions

    return transNorm

def laggedTransCount(lines, k, lag):

    transitions = defaultdict(Counter)

    # if k is zero, just count instances of each state
    if k == 0:
        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1
        return transitions
    

    for seq in lines:
        seqLength = len(seq)
        for t in range(k * lag, seqLength):
            prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
            curr = seq[t] # curr state is at time t
            transitions[prev][curr] += 1  #add 1 count to this prev & curr combination

    return transitions

##### Collapsed Context Markov Sampling #####

def collapsedTransNorm(states, lines, k, smoother):
    
    transitions = defaultdict(Counter)

    #for k=0 version, transition matrix turns into simple marginal state distributions
    if k == 0:
        for state in states:
            transitions[()][state] = smoother #add laplace smoother to each curr state

        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

        total = sum(transitions[()].values())

        return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

    for prev in product(states, repeat=k):

        if hasConsecutiveRepeats(prev): #if prev tuple has consecutive repeats, skip (because this cannot be observed when self-transitions are collapsed)
            continue

        for curr in states:
            transitions[prev][curr] = smoother #add laplace smoother to every valid combination of prev & curr states

    for seq in lines:

        idxs = uniqueIdxs(seq) #find indices of unique states in sequence
        idxsExpanded = expandVec(idxs) #expand indices for mapping

        for t in range(1, len(seq)):

            idxsCollapsed = idxs.index(idxsExpanded[t - 1]) #find corresponding index of each token

            if idxsCollapsed < k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    transNorm = {}

    for cond, counter in transitions.items():
        total = sum(counter.values())
        transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize 

    return transNorm

def collapsedTransCount(lines, k):

    transitions = defaultdict(Counter)

    # if k is zero, just count instances of each state
    if k == 0:
        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1

        return transitions
    
    for seq in lines:
        idxs = uniqueIdxs(seq)
        idxsExpanded = expandVec(idxs)

        for t in range(1, len(seq)):
            idxsCollapsed= idxs.index(idxsExpanded[t - 1])

            if idxsCollapsed< k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    return transitions


In [3]:
#deprecated, version of generation where sub sequence reacts to previous tokens of dom sequence
# def generatePair(length, states, wTrans, bTrans, k, rt, temp):
#     """Generate one (dom, sub) sequence pair.

#     a within-sequence transition matrix is used to generate the dom sequence, 
#     and then a between-sequence transition matrix is used to weight the within-sequence matrix, 
#     and this is used to generate the sub sequence with a reaction time lag to the dom sequence, 
#     simulating reactivity of the sub fish to the dom fish. Temp determines the strength of relationship between dom and sub sequences    
#     """
#     domseq = random.choices(states, k=k) if k > 0 else []
#     subseq = random.choices(states, k=k) if k > 0 else []

#     for t in range(k, length):
#         # dom: only wTrans
#         prev_dom = tuple(domseq[t - k:t])
#         domseq.append(sampleFromDict(wTrans[prev_dom]))

#         # sub: wTrans weighted by bTrans when enough history
#         prev_sub = tuple(subseq[t - k:t])
#         if t < rt or rt == 0:
#             probs = wTrans[prev_sub]
#         else:
#             prev_dom_rt = domseq[t - rt]
#             unnorm = {
#                 curr: wTrans[prev_sub][curr] * (bTrans[curr][prev_dom_rt] ** temp)
#                 for curr in states
#             }
#             Z = sum(unnorm.values())
#             probs = {c: p / Z for c, p in unnorm.items()}
#         subseq.append(sampleFromDict(probs))

#     return domseq, subseq

# def generateAndSaveCorpus(k, rt, n_pairs, temp, alpha,
#                            wTrans, bTrans,
#                            seqlength=500,
#                            states=['a','b','c','d','e','f','g','h','j','k']):
    
#     """Generate n_pairs sequences and split into train/test.

#         k = size of tuple in prev (Markov order used for generating)
#         rt = reaction time between dom and sub
#         n_pairs = number of pairs of sequences generated (x2 for full corpus size since dom and sub)
#         temp = strength of reactivity between dom and sub (weighting for bTrans when multiplied with wTrans)
#         alpha = dirichlet alpha determines the peakiness of distribution, approximation of entropy of system
#         wTrans = transition matrix used for dom (dom assumed to be non-reactive)
#         bTrans = transition matrix for sub
#         seqlength = length of sequences
#         states = unique states used within sequences

#     Returns a dict with sequence lists and metadata (no files written)."""

#     pairs = [
#         generatePair(length=seqlength, states=states,
#                      wTrans=wTrans, bTrans=bTrans, k=k, rt=rt, temp=temp)
#         for _ in range(n_pairs)
#     ]
#     train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

#     def pairs_to_seqs(pair_list):
#         dom_seqs, sub_seqs = [], []
#         for dom, sub in pair_list:
#             dom_seqs.append(list(dom))
#             sub_seqs.append(list(sub))
#         return dom_seqs, sub_seqs

#     train_dom, train_sub = pairs_to_seqs(train_pairs)
#     test_dom,  test_sub  = pairs_to_seqs(test_pairs)
#     mean_entropy = nonselfMeanNormEntropy(wTrans, len(states))

#     return {
#         'train_dom': train_dom, 'train_sub': train_sub,
#         'test_dom':  test_dom,  'test_sub':  test_sub,
#         'params': {'k': k, 'rt': rt, 'n_pairs': n_pairs, 'temp': temp, 'alpha': alpha},
#         'mean_entropy': mean_entropy,
#     }



# def analyze_corpus(corpus_data, kmax, smoothers, lagmax, states,
#                    way='semicol'):
    
#     """Fit k-order models (lagged or semicol) on train sequences and score on test.

#     Parameters
#     ----------
#     corpus_data : dict  Output of generateAndSaveCorpus.
#     kmax        : int   Maximum order to test.
#     smoothers   : list  Laplace-style smoother values.
#     lagmax      : int   Maximum lag (only used when way='lagged').
#     states      : list  Alphabet.
#     way         : str   'semicol' or 'lagged'.

#     Returns
#     -------
#     pd.DataFrame with one row per (k, smoother[, lag]) combination.
#     """
#     train_lines = corpus_data['train_dom']
#     test_lines  = corpus_data['test_dom']
#     results = []

#     for k in range(0, kmax + 1):
#         for smoother in smoothers:
#             if way == 'lagged':
#                 for lag in [1] + list(range(10, lagmax + 1, 10)):
#                     train_trans = normalized_lagged_transitions(
#                         states, lines=train_lines, k=k, smoother=smoother, lag=lag)
#                     test_trans  = unnormalized_lagged_transitions(
#                         lines=test_lines, k=k, lag=lag)
#                     ll = average_ll(train_trans, test_trans)
#                     results.append({'k': k, 'smoother': smoother, 'lag': lag,
#                                     'log_likelihood': ll})
#             else:  # semicol
#                 train_trans = normalize_semicol_transitions(
#                     states, lines=train_lines, k=k, smoother=smoother)
#                 test_trans  = unnormalized_semicol_transitions(
#                     lines=test_lines, k=k)
#                 ll = average_ll(train_trans, test_trans)
#                 results.append({'k': k, 'smoother': smoother,
#                                 'log_likelihood': ll})

#     return pd.DataFrame(results)


# def runOnce(alpha_values, fixed_params,
#             kmax=5, smoothers=[0.01], lagmax=60,
#             states=['a','b','c','d','e','f','g','h','j','k'],
#             dur_prob=0.95, seqlength=500,
#             true_k_values=[0, 1, 2, 3, 4], rt=5,
#             way='semicol'):
#     """Run one sweep over alpha × true_k combinations.

#     Returns
#     -------
#     all_results  : dict  (alpha, true_k) -> results_df
#     entropy_map  : dict  (alpha, true_k) -> mean_entropy



# to do: n states should vary so that this can be adapted for bird data as well


#     """
#     all_results, entropy_map = {}, {}
#     bTrans_cache, wTrans_cache = {}, {}
#     total_runs = len(alpha_values) * len(true_k_values)
#     current_run = 0

#     for alpha in alpha_values:
#         for true_k in true_k_values:
#             current_run += 1
#             print(f"  Run {current_run}/{total_runs}: alpha={alpha}, true_k={true_k}")

#             params  = {**fixed_params, 'alpha': alpha}
#             temp    = params['temperature']
#             n_pairs = params['n_pairs']

#             # bTrans: shared per alpha
#             if alpha not in bTrans_cache:
#                 bTrans = defaultdict(Counter)
#                 for prev in states:
#                     weights = np.random.dirichlet([1] * len(states))
#                     for curr, w in zip(states, weights):
#                         bTrans[curr][prev] = w
#                 bTrans_cache[alpha] = bTrans
#             else:
#                 bTrans = bTrans_cache[alpha]

#             # wTrans: one per (alpha, true_k)
#             cache_key = (alpha, true_k)
#             if cache_key not in wTrans_cache:
#                 wTrans = defaultdict(Counter)
#                 for prev in product(states, repeat=true_k):
#                     if true_k == 0:
#                         dirichlet_sample = np.random.dirichlet([alpha] * len(states))
#                         for s, p in zip(states, dirichlet_sample):
#                             wTrans[prev][s] = p
#                     else:
#                         last_state = prev[-1]
#                         other_states = [s for s in states if s != last_state]
#                         dirichlet_sample = np.random.dirichlet(
#                             [alpha] * len(other_states)) * (1.0 - dur_prob)
#                         wTrans[prev][last_state] = dur_prob
#                         for s, p in zip(other_states, dirichlet_sample):
#                             wTrans[prev][s] = p
#                 wTrans_cache[cache_key] = wTrans
#             else:
#                 wTrans = wTrans_cache[cache_key]

#             corpus_data = generateAndSaveCorpus(
#                 k=true_k, rt=rt, n_pairs=n_pairs, temp=temp, alpha=alpha,
#                 seqlength=seqlength, states=states,
#                 wTrans=wTrans, bTrans=bTrans)

#             results_df = analyze_corpus(
#                 corpus_data, kmax=kmax, smoothers=smoothers,
#                 lagmax=lagmax, states=states, way=way)

#             all_results[cache_key]  = results_df
#             entropy_map[cache_key]  = corpus_data['mean_entropy']

#             if not results_df.empty:
#                 best_idx = results_df['log_likelihood'].idxmax()
#                 best_k   = results_df.loc[best_idx, 'k']
#                 best_ll  = results_df.loc[best_idx, 'log_likelihood']
#                 print(f"    Best detected k={best_k} (LL={best_ll:.4f}), "
#                       f"entropy={corpus_data['mean_entropy']:.4f}")

#     return all_results, entropy_map

# def repeated_sims(param_values, fixed_params,
#                   kmax=5, smoothers=[0.01], lagmax=60,
#                   n_simulations=20, dur_prob=0.95,
#                   seqlength=500, true_k_values=None, rt=5,
#                   way='semicol'):
#     """Run repeated_sims simulations and return a flat DataFrame of outcomes.

#     Parameters
#     ----------
#     param_values    : list  alpha values to sweep.
#     fixed_params    : dict  Must contain 'n_pairs', 'temperature'.
#     way             : str   'semicol' or 'lagged' — detection method to use.
#     """
#     if true_k_values is None:
#         true_k_values = [0, 1, 2, 3, 4]

#     rows = []
#     for i in range(n_simulations):
#         print(f"\n{'='*50}")
#         print(f"Simulation {i+1}/{n_simulations}  [way={way}]")
#         print(f"{'='*50}")
#         all_results, entropy_map = runOnce(
#             param_values, fixed_params,
#             kmax=kmax, smoothers=smoothers, lagmax=lagmax,
#             dur_prob=dur_prob, seqlength=seqlength,
#             true_k_values=true_k_values, rt=rt, way=way)

#         for (alpha, true_k), df in all_results.items():
#             if df is None or df.empty:
#                 continue
#             best_idx = df['log_likelihood'].idxmax()
#             best_k   = df.loc[best_idx, 'k']
#             best_ll  = df.loc[best_idx, 'log_likelihood']
#             rows.append({
#                 'sim':      i,
#                 'alpha':    alpha,
#                 'n_pairs':  fixed_params['n_pairs'],
#                 'true_k':   true_k,
#                 'best_k':   best_k,
#                 'best_ll':  best_ll,
#                 'correct':  int(best_k == true_k),
#                 'entropy':  entropy_map.get((alpha, true_k), np.nan),
#                 'way':      way,
#             })

#     return pd.DataFrame(rows)

# def repeated_sims_vary_n(n_pairs_values, fixed_params_base,
#                           alpha_values,
#                           kmax=5, smoothers=[0.01], lagmax=60,
#                           n_simulations=20, dur_prob=0.95,
#                           seqlength=500, true_k_values=None, rt=5,
#                           way='semicol'):
#     """Run repeated_sims for each n_pairs value.

#     Returns a combined DataFrame with an 'n_pairs' column.
#     """
#     if true_k_values is None:
#         true_k_values = [0, 1, 2, 3, 4]

#     all_dfs = []
#     for n_pairs in n_pairs_values:
#         print(f"\n{'#'*60}")
#         print(f"n_pairs = {n_pairs}  [way={way}]")
#         print(f"{'#'*60}")
#         fixed_params = {**fixed_params_base, 'n_pairs': n_pairs}
#         df = repeated_sims(
#             alpha_values, fixed_params,
#             kmax=kmax, smoothers=smoothers, lagmax=lagmax,
#             n_simulations=n_simulations, dur_prob=dur_prob,
#             seqlength=seqlength, true_k_values=true_k_values,
#             rt=rt, way=way)
#         # n_pairs already written per-row by repeated_sims
#         all_dfs.append(df)
#     return pd.concat(all_dfs, ignore_index=True)



In [9]:
# SIMULATIONS #

## Helper Functions ##

### entropy calculations ###

def entropy(prob_dist):

    #Shannon entropy calculation
    probs = np.array(list(prob_dist.values()))
    probs = probs[probs > 0]

    return -np.sum(probs * np.log(probs))

def nonselfEntropy(prob_dist, context):

    #non-self-transition normalized entropy
    if len(context) == 0:
        return entropy(prob_dist)
    
    last_state = context[-1]
    non_self_dist = {s: p for s, p in prob_dist.items() if s != last_state}

    if not non_self_dist:
        return 0.0
    
    total = sum(non_self_dist.values())

    if total == 0:
        return 0.0
    
    normalized = {s: p / total for s, p in non_self_dist.items()}

    return entropy(normalized)

def nonselfMeanNormEntropy(transitions_dict, num_states):

    #mean non-self entropy across transition matrix
    entropies = []

    for context, prob_dist in transitions_dict.items():
        if not prob_dist:
            continue

        h = nonselfEntropy(prob_dist, context)
        max_entropy = np.log(num_states - 1) if len(context) > 0 else np.log(num_states)
        normalized_entropy = h / max_entropy if max_entropy > 0 else 0.0
        entropies.append(normalized_entropy)
        
    return np.mean(entropies) if entropies else np.nan

### synthetic dataset generation ###

def sampleFromDict(probs):

    states = list(probs.keys())

    weights = list(probs.values())

    return random.choices(states, weights=weights, k=1)[0]

def generateSeq(length, states, trans, k):

    seq = random.choices(states, k = k) if k > 0 else []

    for t in range(k, length):
        prev = tuple(seq[t - k:t])
        #prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k)))
        seq.append(sampleFromDict(trans[prev]))

    return seq

def generateCorpus(k, nPairs, alpha, trans, seqLength, states):

    seqs = [generateSeq(length = seqLength, states = states, trans = trans, k=k) for _ in range(nPairs)]

    trainSeqs, testSeqs = train_test_split(seqs, test_size=0.2, random_state=random_state)

    meanH = nonselfMeanNormEntropy(trans, len(states))

    return {
        'train': trainSeqs,
        'test':  testSeqs, 
        'params': {'k': k, 'nPairs': nPairs, 'alpha': alpha},
        'meanH': meanH,
    }

def analyzeCorpus(corpus, kMax, smootherSet, lagSet, states, way: Way):

    trainSeqs = corpus['train']
    testSeqs = corpus['test']
                      
    results = []

    for k in range(0, kMax + 1):
        for smoother in smootherSet:
            if way == 'lagged':
                for lag in lagSet:

                    train_trans = laggedTransNorm(
                        states, lines=trainSeqs, k=k, smoother=smoother, lag=lag)
                    
                    test_trans  = laggedTransCount(
                        lines=testSeqs, k=k, lag=lag)
                    
                    ll = averageLL(train_trans, test_trans)

                    results.append({'k': k, 'smoother': smoother, 'lag': lag,
                                    'log_likelihood': ll})
                    
            else:  # semicol
                train_trans = collapsedTransNorm(
                    states, lines=trainSeqs, k=k, smoother=smoother)
                
                test_trans  = collapsedTransCount(
                    lines=testSeqs, k=k)
                
                ll = averageLL(train_trans, test_trans)

                results.append({'k': k, 'smoother': smoother,
                                'log_likelihood': ll})

    return pd.DataFrame(results)

### simulation runner ###

""" pipeline: 

for each combination of true generative k, dirichlet alpha, corpus size, and unique state length

- generate transition matrix with certain peakiness/entropy
- generate corpus of sequences based on that transition matrix
- analyse each corpus and calculate how often true generative k matches detected k

 """

def simulate(nSims, #sim parameters
             trueKs, nStatesSet, alphaSet, corpusSizes, selfTransProb, seqLength, #generative params
             kMax2Detect, smootherSet, lagSet, ways): #detection params
    
    rows = []

    for i in range(1, nSims+1):

        print(f"Sim {i}")

        transCache = {}

        for trueK in trueKs:

            for nStates in nStatesSet: #start at binary string

                states = list(string.ascii_lowercase[:nStates])

                for alpha in alphaSet:

                    for corpusSize in corpusSizes:

                        for way in ways:

                            cache_key = (trueK, nStates, alpha)

                            if cache_key not in transCache:

                                trans = defaultdict(Counter)

                                for prev in product(states, repeat=trueK):
                                    if trueK == 0:
                                        dirichlet_sample = np.random.dirichlet([alpha] * len(states))
                                        for s, p in zip(states, dirichlet_sample):
                                            trans[prev][s] = p

                                    else:
                                        last_state = prev[-1]
                                        other_states = [s for s in states if s != last_state]
                                        dirichlet_sample = np.random.dirichlet(
                                            [alpha] * len(other_states)) * (1.0 - selfTransProb)
                                        trans[prev][last_state] = selfTransProb
                                        for s, p in zip(other_states, dirichlet_sample):
                                            trans[prev][s] = p

                                transCache[cache_key] = trans

                            else:
                                trans = transCache[cache_key]

                            corpus = generateCorpus(k=trueK, nPairs=corpusSize, alpha=alpha, trans = trans, seqLength=seqLength, states=states)

                            results = analyzeCorpus(corpus = corpus, kMax = kMax2Detect, smootherSet = smootherSet, lagSet = lagSet, states = states, way = way)

                            results = results.copy()
                            results['sim'] = i
                            results['alpha'] = alpha
                            results['corpusSize'] = corpusSize
                            results['trueK'] = trueK
                            results['nStates'] = nStates                            
                            results['way'] = way
                            results['entropy'] = corpus['meanH']

                            rows.append(results)


    df = pd.concat(rows, ignore_index=True)

    return pd.DataFrame(df)

## Do It! ##

# params
nSims = 25
trueKs = [1, 2, 3, 4, 5]
nStatesSet = [2, 4, 6, 8, 10, 12]
alphaSet = [0.1, .5, 1, 10]
corpusSizes = [50, 100, 250, 500, 1000]
selfTransProb = .95
seqLength = 250
kMax2Detect = 5 
smootherSet = [.1]
lagSet = [1, 10, 25, 50]
ways= ['semicol', 'lagged']

#simulate and save results
df = simulate(nSims, #sim parameters
             trueKs, nStatesSet, alphaSet, corpusSizes, selfTransProb, seqLength, #generative params
             kMax2Detect, smootherSet, lagSet, ways)

df.to_csv('simResults.csv')


Sim 1
Sim 2
Sim 3
Sim 4
Sim 5
Sim 6
Sim 7
Sim 8
Sim 9
Sim 10
Sim 11
Sim 12
Sim 13
Sim 14
Sim 15
Sim 16
Sim 17
Sim 18
Sim 19
Sim 20
Sim 21
Sim 22
Sim 23
Sim 24
Sim 25
